# 01 Data Cleaning

This notebook validates the raw Kaggle Customer Personality Analysis dataset and applies the same conservative cleaning workflow implemented in `src/data_cleaning.py`.

Scope for this step:
- Load and inspect the raw CSV.
- Check missing values, duplicate rows, dates, and impossible values.
- Save a cleaned dataset for later EDA, clustering, and supervised modeling.

This notebook does not build models, create targets, cluster customers, scale variables, or perform advanced feature engineering.

## Setup

The project code lives in `src/`, while this notebook lives in `notebooks/`. The setup below finds the project root and imports the reusable cleaning functions.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from data_cleaning import (
    CLEAN_DATA_PATH,
    MAX_REASONABLE_AGE_AT_SIGNUP,
    RAW_DATA_PATH,
    clean_customer_data,
    load_raw_data,
    save_clean_data,
    standardize_column_names,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Clean data path: {CLEAN_DATA_PATH}")

Project root: /Users/arjun/Documents/Intro DS final
Raw data path: /Users/arjun/Documents/Intro DS final/data/raw/marketing_campaign.csv
Clean data path: /Users/arjun/Documents/Intro DS final/data/processed/customer_clean.csv


## Load the Raw Dataset

The Kaggle file is named like a CSV, but the version used here is tab-delimited. The loader detects the delimiter before reading with pandas.

In [2]:
raw_df = load_raw_data(RAW_DATA_PATH)

print(f"Raw dataframe shape: {raw_df.shape}")
print("\nColumn names:")
print(raw_df.columns.tolist())

display(raw_df.head())

Raw dataframe shape: (2240, 29)

Column names:
['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


## Raw Structure Checks

Before changing any values, inspect data types, missing values, and basic summary statistics.

In [3]:
dtype_table = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": [str(dtype) for dtype in raw_df.dtypes],
})
display(dtype_table)

missing_counts = raw_df.isna().sum().sort_values(ascending=False)
display(missing_counts.to_frame("missing_values"))

,column,dtype
0,ID,int64
1,Year_Birth,int64
2,Education,str
3,Marital_Status,str
4,Income,float64
5,Kidhome,int64
6,Teenhome,int64
7,Dt_Customer,str
8,Recency,int64
9,MntWines,int64


,missing_values
Income,24
ID,0
NumDealsPurchases,0
Z_Revenue,0
Z_CostContact,0
Complain,0
AcceptedCmp2,0
AcceptedCmp1,0
AcceptedCmp5,0
AcceptedCmp4,0


In [4]:
print("Numeric summary statistics")
display(raw_df.describe(include="number").T)

print("Categorical/date summary statistics")
display(raw_df.describe(include=["object", "string"]).T)

Numeric summary statistics


,count,mean,std,min,25%,50%,75%,max
ID,2240.0,5592.159821,3246.662198,0.0,2828.25,5458.5,8427.75,11191.0
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0


Categorical/date summary statistics


,count,unique,top,freq
Education,2240,5,Graduation,1127
Marital_Status,2240,8,Married,864
Dt_Customer,2240,663,31-08-2012,12


## Duplicate Rows and Missing Income

`Income` is the only field with missing values. Because income will be important for customer profiling and later modeling, the missing rows need a clear decision rather than being ignored.

In [5]:
duplicate_rows = raw_df.duplicated().sum()
missing_income = raw_df[raw_df["Income"].isna()]

print(f"Duplicate full rows: {duplicate_rows}")
print(f"Rows with missing Income: {len(missing_income)}")
print(f"Missing Income share: {len(missing_income) / len(raw_df):.2%}")

display(missing_income[["ID", "Education", "Marital_Status", "Kidhome", "Teenhome", "Dt_Customer"]].head(10))

Duplicate full rows: 0
Rows with missing Income: 24
Missing Income share: 1.07%


,ID,Education,Marital_Status,Kidhome,Teenhome,Dt_Customer
10,1994,Graduation,Married,1,0,15-11-2013
27,5255,Graduation,Single,1,0,20-02-2013
43,7281,PhD,Single,0,0,05-11-2013
48,7244,Graduation,Single,2,1,01-01-2014
58,8557,Graduation,Single,1,0,17-06-2013
71,10629,2n Cycle,Married,1,0,14-09-2012
90,8996,PhD,Married,2,1,19-11-2012
91,9235,Graduation,Single,1,1,27-05-2014
92,5798,Master,Together,0,0,23-11-2013
128,8268,PhD,Married,0,1,11-07-2013


In [6]:
print("Missing Income by Education")
display(missing_income["Education"].value_counts(dropna=False).to_frame("rows"))

print("Missing Income by Marital Status")
display(missing_income["Marital_Status"].value_counts(dropna=False).to_frame("rows"))

Missing Income by Education


,rows
Education,
Graduation,11
PhD,5
Master,5
2n Cycle,3


Missing Income by Marital Status


,rows
Marital_Status,
Single,9
Married,7
Together,7
Widow,1


## Date Parsing and Suspicious Value Checks

The cleaning script standardizes columns to `snake_case` and parses `Dt_Customer` as a real datetime value. The checks below look for clearly invalid values only, not ordinary outliers that should be studied later.

In [7]:
preview_df = standardize_column_names(raw_df)
preview_df["dt_customer"] = pd.to_datetime(preview_df["dt_customer"], format="%d-%m-%Y", errors="coerce")

count_columns = [column for column in preview_df.columns if column.startswith("num_")]
amount_columns = [column for column in preview_df.columns if column.startswith("mnt_")]

signup_year = preview_df["dt_customer"].dt.year
unrealistic_birth_mask = (
    (preview_df["year_birth"] > signup_year)
    | ((signup_year - preview_df["year_birth"]) > MAX_REASONABLE_AGE_AT_SIGNUP)
)

checks = {
    "date_parse_failures": int(preview_df["dt_customer"].isna().sum()),
    "negative_income_rows": int((preview_df["income"] < 0).sum()),
    "unrealistic_birth_year_rows": int(unrealistic_birth_mask.sum()),
    "negative_purchase_count_rows": int((preview_df[count_columns] < 0).any(axis=1).sum()),
    "negative_spending_amount_rows": int((preview_df[amount_columns] < 0).any(axis=1).sum()),
}

display(pd.Series(checks, name="count").to_frame())
print(f"Customer date range: {preview_df['dt_customer'].min().date()} to {preview_df['dt_customer'].max().date()}")

,count
date_parse_failures,0
negative_income_rows,0
unrealistic_birth_year_rows,3
negative_purchase_count_rows,0
negative_spending_amount_rows,0


Customer date range: 2012-07-30 to 2014-06-29


In [8]:
print("Rows with unrealistic birth years")
display(
    preview_df.loc[
        unrealistic_birth_mask,
        ["id", "year_birth", "education", "marital_status", "income", "dt_customer"],
    ]
)

print("Income range before cleaning")
print(preview_df["income"].min(), "to", preview_df["income"].max())

Rows with unrealistic birth years


,id,year_birth,education,marital_status,income,dt_customer
192,7829,1900,2n Cycle,Divorced,36640.0,2013-09-26
239,11004,1893,2n Cycle,Single,60182.0,2014-05-17
339,1150,1899,PhD,Together,83532.0,2013-09-26


Income range before cleaning
1730.0 to 666666.0


## Cleaning Decisions

Cleaning is intentionally conservative at this stage:

- Standardize column names to `snake_case` so future scripts and notebooks are easier to read.
- Remove exact duplicate rows if present.
- Parse `dt_customer` using the day-month-year format in the raw file.
- Drop rows with missing `income`; there are only 24 such rows, about 1% of the raw data, and income is central to this project.
- Remove clearly unrealistic birth years where the customer would be over 100 at signup or born after signup.
- Remove negative income, purchase count, or spending values if any appear.
- Keep high but possible values, such as the very large income maximum, for later EDA/outlier discussion rather than removing them now.

In [9]:
cleaned_df, cleaning_report = clean_customer_data(raw_df)
save_clean_data(cleaned_df, CLEAN_DATA_PATH)

display(pd.Series(cleaning_report, name="count").to_frame())
print(f"Saved cleaned dataset to: {CLEAN_DATA_PATH}")

,count
starting_rows,2240
starting_columns,29
duplicate_rows_removed,0
invalid_date_rows_removed,0
missing_income_rows_removed,24
negative_income_rows_removed,0
unrealistic_birth_year_rows_removed,3
negative_purchase_count_rows_removed,0
negative_spending_amount_rows_removed,0
ending_rows,2213


Saved cleaned dataset to: /Users/arjun/Documents/Intro DS final/data/processed/customer_clean.csv


## Cleaned Dataset Validation

The final checks confirm that the cleaned file has no missing values, no duplicate full rows, valid parsed dates, and no clearly impossible values targeted by this cleaning step.

In [10]:
print(f"Cleaned dataframe shape: {cleaned_df.shape}")

remaining_missing = cleaned_df.isna().sum()
display(remaining_missing.to_frame("missing_values"))

validation_checks = {
    "duplicate_full_rows": int(cleaned_df.duplicated().sum()),
    "missing_values_total": int(remaining_missing.sum()),
    "date_parse_failures": int(cleaned_df["dt_customer"].isna().sum()),
    "negative_income_rows": int((cleaned_df["income"] < 0).sum()),
    "negative_purchase_count_rows": int((cleaned_df[count_columns] < 0).any(axis=1).sum()),
    "negative_spending_amount_rows": int((cleaned_df[amount_columns] < 0).any(axis=1).sum()),
}
display(pd.Series(validation_checks, name="count").to_frame())

Cleaned dataframe shape: (2213, 29)


,missing_values
id,0
year_birth,0
education,0
marital_status,0
income,0
kidhome,0
teenhome,0
dt_customer,0
recency,0
mnt_wines,0


,count
duplicate_full_rows,0
missing_values_total,0
date_parse_failures,0
negative_income_rows,0
negative_purchase_count_rows,0
negative_spending_amount_rows,0


In [11]:
display(cleaned_df.head())

assert cleaned_df.isna().sum().sum() == 0
assert cleaned_df.duplicated().sum() == 0
assert cleaned_df["dt_customer"].notna().all()
assert (cleaned_df["income"] < 0).sum() == 0
assert (cleaned_df[count_columns] < 0).any(axis=1).sum() == 0
assert (cleaned_df[amount_columns] < 0).any(axis=1).sum() == 0

,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,mnt_wines,mnt_fruits,mnt_meat_products,mnt_fish_products,mnt_sweet_products,mnt_gold_prods,num_deals_purchases,num_web_purchases,num_catalog_purchases,num_store_purchases,num_web_visits_month,accepted_cmp3,accepted_cmp4,accepted_cmp5,accepted_cmp1,accepted_cmp2,complain,z_cost_contact,z_revenue,response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0
